# Mixed Precision Training — Micikevicius et al. (2017)

_Papers · Efficiency & Compression_

**Train in 16-bit floats for speed and memory, but keep a 32-bit master weight copy and scale the loss so tiny gradients do not vanish.**

---

This notebook is a practice scaffold for the **Mixed Precision Training — Micikevicius et al. (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch

torch.manual_seed(0)

# --- 0a. Loss scaling rescues a small gradient (paper Section 3.2). ---
g = torch.tensor(1e-8, dtype=torch.float32)
print("gradient cast straight to FP16:", g.half().item(), " (underflowed to 0)")
S = 1024.0
scaled16 = (g * S).half()                 # scale in FP32, THEN cast to FP16
print("scaled by 1024 then cast to FP16:", scaled16.item(), " (survived)")
print("unscaled back in FP32:", (scaled16.float() / S).item())
print("smallest positive FP16 (2^-24):", torch.tensor(2.0**-24, dtype=torch.float16).item())
# gradient cast straight to FP16: 0.0  (underflowed to 0)
# scaled by 1024 then cast to FP16: 1.0251998901367188e-05  (survived)
# unscaled back in FP32: 1.0011717677116394e-08
# smallest positive FP16 (2^-24): 5.960464477539063e-08

# --- 0b. FP32 master copy rescues a tiny weight update (paper Section 3.1). ---
w16 = torch.tensor(256.0, dtype=torch.float16)   # pure-FP16 weight
w32 = torch.tensor(256.0, dtype=torch.float32)   # FP32 master copy
update = torch.tensor(-0.009, dtype=torch.float32)
for _ in range(200):
    w16 = (w16 + update.half())            # pure FP16: update right-shifts to nothing
    w32 = w32 + update                     # FP32 master: tiny updates accumulate
print("pure-FP16 weight after 200 steps:", w16.item(), " (frozen at 256.0)")
print("FP32-master weight after 200 steps:", round(w32.item(), 3), " (moved)")
# pure-FP16 weight after 200 steps: 256.0  (frozen at 256.0)
# FP32-master weight after 200 steps: 254.2  (moved)


# --- 1. Train a tiny linear regression two ways: pure FP16 vs FP32 master copy. ---
# Weights start near 256, where the FP16 spacing (0.25) dwarfs each per-step update,
# so the pure-FP16 weights cannot move while the FP32-master weights can.
n, d = 64, 4
gen = torch.Generator().manual_seed(1)
X = torch.randn(n, d, generator=gen) * 0.1
w_true = torch.full((d, 1), 250.0)
y = X @ w_true

def train(use_master, steps=200, lr=5e-2):
    w32 = torch.full((d, 1), 256.0)        # FP32 master copy
    w16 = w32.half().clone()               # FP16 copy used in the forward pass
    losses = []
    for t in range(steps):
        w = (w32 if use_master else w16).float()
        err = X @ w - y
        losses.append((err ** 2).mean().item())
        grad = (2.0 / n) * (X.t() @ err)   # FP32 gradient
        if use_master:
            w32 = w32 - lr * grad          # accumulate the update in FP32
            w16 = w32.half()               # round to FP16 for the next pass
        else:
            w16 = (w16 + (-lr * grad).half())   # pure FP16: update vanishes on add
    return losses

pure   = train(use_master=False)
master = train(use_master=True)
print("\nPure-FP16   loss:  start", round(pure[0], 3),   " end", round(pure[-1], 3))
print("FP32-master loss:  start", round(master[0], 3), " end", round(master[-1], 3))
# Pure-FP16   loss:  start 1.988  end 1.988   (flat -- weights frozen, no learning)
# FP32-master loss:  start 1.988  end 1.145   (fell -- the master copy let it learn)
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_With tiny per-step updates, does the training loss fall when weights are kept in an FP32 master copy vs updated directly in FP16?_

In [ ]:
import torch, numpy as np

torch.manual_seed(0)
# Tiny linear regression. Weights start at 256.0, where consecutive FP16 numbers are
# 0.25 apart. Each per-step update (lr*grad ~ 0.009) is well below that spacing, so in
# PURE FP16 every update right-shifts to zero on the add and the loss stays flat. An
# FP32 MASTER copy accumulates the same updates and learns (paper Sections 3.1).
n, d = 64, 4
g = torch.Generator().manual_seed(1)
X = torch.randn(n, d, generator=g) * 0.1
w_true = torch.full((d, 1), 250.0)
y = X @ w_true

def train(use_master, steps=200, lr=5e-2):
    w32 = torch.full((d, 1), 256.0)        # FP32 master copy
    w16 = w32.half().clone()
    losses = []
    for t in range(steps):
        w = (w32 if use_master else w16).float()
        err = X @ w - y
        losses.append((err ** 2).mean().item())
        grad = (2.0 / n) * (X.t() @ err)
        if use_master:
            w32 = w32 - lr * grad          # accumulate in FP32
            w16 = w32.half()               # round to FP16 for the next forward
        else:
            w16 = (w16 + (-lr * grad).half())   # pure FP16: update vanishes
    return losses

pure   = train(use_master=False)
master = train(use_master=True)
idx = np.linspace(0, 199, 30).astype(int)
print("Pure FP16  :", [[int(i), round(pure[i], 3)] for i in idx])
print("FP32 master:", [[int(i), round(master[i], 3)] for i in idx])
# Pure FP16  -> flat at 1.988 (weights frozen at 256.0).
# FP32 master -> falls 1.988 -> 1.145 over 200 steps.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The loss-scaling ablation. A gradient $g = 3\times10^{-8}$ is computed in a network
            trained in FP16. (a) What does it become when stored in FP16 with no loss scaling? (b) With a
            scale factor $S = 1024$, what does the stored value become, and what do you recover after
            unscaling? (c) What does this demonstrate about loss scaling?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compare $g$ to the FP16 floor $2^{-24} \approx 5.96\times10^{-8}$. Since $3\times10^{-8} \lt 5.96\times10^{-8}$, storing $g$ in FP16 gives $0.0$. — _Anything below $2^{-24}$ underflows to exactly zero in FP16 (&sect;3.1) &mdash; the learning signal is lost._
- Scale first: $g \times 1024 = 3.07\times10^{-5}$, which is well inside the FP16 range, so it stores as a nonzero value. — _Multiplying by $S$ shifts the small gradient up above the underflow floor before it is cast to FP16._
- Unscale: divide the stored value by $1024$ to get back $\approx 3\times10^{-8}$. — _Unscaling restores the true gradient magnitude so the optimizer step is the same size as in FP32 training._

**Answer:** (a) With no scaling, $3\times10^{-8}$ is below $2^{-24}$, so FP16 stores it as $0.0$
                 &mdash; the gradient vanishes. (b) Scaled by $1024$ it becomes $3.07\times10^{-5}$ (nonzero in
                 FP16); unscaling gives back $\approx 3\times10^{-8}$. (c) Loss scaling rescues exactly the
                 small gradients that would otherwise underflow to zero, at the cost of one multiply and one
                 divide. This is the &sect;3.2 mechanism.

</details>

**Problem 2.** The master-copy ablation. A weight sits at $256.0$; near $256$ consecutive FP16 numbers
            are $0.25$ apart. The optimizer wants to subtract $0.009$ each step. After $200$ steps, where is
            the weight (i) if you update it directly in FP16, and (ii) if you keep an FP32 master copy? What
            does the gap show?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compare the update $0.009$ to the FP16 spacing $0.25$ near $256$. Since $0.009 \lt 0.25$, adding it in FP16 right-shifts it past the last kept bit and the weight does not change. — _When the weight is more than $\sim2048$ times the update, the addition leaves the weight unchanged (&sect;3.1)._
- Pure FP16: the weight stays $256.0$ for all $200$ steps &mdash; frozen, no learning. — _Every step the same right-shift erases the update, so nothing accumulates._
- FP32 master: the $0.009$ updates accumulate in FP32; after $200$ steps the weight has moved to about $254.65$. — _FP32 keeps enough bits that the small update survives the add and accumulates over steps._

**Answer:** (i) In pure FP16 the weight is still $256.0$ after $200$ steps &mdash; the update is
                 smaller than the FP16 spacing, so every add is a no-op and learning stalls. (ii) With the
                 FP32 master copy the weight reaches about $254.65$. Same updates, same learning rate;
                 only the precision of the accumulator differs. This isolates the master copy as the reason
                 tiny updates take effect &mdash; the &sect;3.1 result reproduced on one number.

</details>

**Problem 3.** You enable loss scaling with $S = 1024$ but forget to divide the gradients by $S$ before calling
            optimizer.step(). The loss explodes to nan within a few steps. Why,
            and what is the one-line fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Trace the update size: the gradients are still $1024\times$ too large, so each update is $1024\times$ bigger than intended. — _Scaling the loss scaled every gradient by $S$; without unscaling, that factor flows straight into the optimizer step._
- A $1024\times$ learning rate overshoots wildly, the weights blow up, and the loss becomes inf then nan. — _Hugely oversized steps diverge instead of descending._
- Insert the unscale right after backward, before the step: divide every gradient by $S$ (or call scaler.unscale_(optimizer)). — _That restores the true gradient magnitude, so the update matches FP32 training (&sect;3.2)._

**Answer:** Loss scaling multiplied every gradient by $S = 1024$; skipping the unscale leaves the
                 updates $1024\times$ too large, so the optimizer overshoots and the loss diverges to
                 nan. Fix: unscale the gradients (divide by $S$) immediately after the
                 backward pass and before optimizer.step() &mdash; exactly what
                 GradScaler automates.

</details>